# Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import f1_score, classification_report

# Importing the Dataset

In [2]:
train_data = pd.read_csv("/content/Corona_NLP_train.csv", encoding='latin-1')
test_data = pd.read_csv("/content/Corona_NLP_test.csv", encoding='latin-1')

# Preprocessing Dataset

In [3]:
train_data = train_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
test_data = test_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)

In [4]:
le = preprocessing.LabelEncoder()
le.fit(train_data['Sentiment'])
train_data['Sentiment'] = le.transform(train_data['Sentiment'])
test_data['Sentiment'] = le.transform(test_data['Sentiment'])

In [5]:
X_train = train_data['OriginalTweet']
y_train = train_data['Sentiment']
X_test = test_data['OriginalTweet']
y_test = test_data['Sentiment']

In [6]:
y_train = to_categorical(y_train, num_classes=5)
y_test = to_categorical(y_test, num_classes=5)

# Compute text Statistics for Vectorization

In [7]:
averageWordsLength = round(sum([len(i.split()) for i in train_data['OriginalTweet']]) / len(train_data['OriginalTweet']))
totalWordsLength = len(set(" ".join(train_data['OriginalTweet']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 41157
Average words per message: 31
Approximate vocabulary size: 136386


# Model development

In [8]:
model = keras.Sequential()
model.add(layers.Input(shape=(1,), dtype=tf.string))

In [9]:
text_vectorizer = layers.TextVectorization(
    max_tokens=20000,#totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=50#averageWordsLength
)
text_vectorizer.adapt(X_train)

model.add(text_vectorizer)

In [10]:
vocab_size = text_vectorizer.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))
embedding_index = {}

with open('glove.6B.50d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_dim=50
embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if i < len(vocab_size):
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [11]:
embedding_layer = layers.Embedding(
    input_dim=20000,#totalWordsLength,
    output_dim=embedding_dim,
    trainable=False,
    weights=[embedding_matrix]
)
model.add(embedding_layer)

In [12]:
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.LSTM(64))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(5, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 50)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 50, 50)         │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 50, 64)         │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,066,949 (4.07 MB)

 Trainable params: 66,949 (261.52 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

# Model training

In [13]:
history = model.fit(X_train.to_numpy(), y_train, epochs=20, batch_size=64, validation_data=(X_test.to_numpy(), y_test))

Epoch 1/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - accuracy: 0.3228 - loss: 1.4750 - val_accuracy: 0.4713 - val_loss: 1.2575
Epoch 2/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.4693 - loss: 1.2292 - val_accuracy: 0.4853 - val_loss: 1.2136
Epoch 3/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.5064 - loss: 1.1647 - val_accuracy: 0.5029 - val_loss: 1.1930
Epoch 4/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.5271 - loss: 1.1276 - val_accuracy: 0.5258 - val_loss: 1.1190
Epoch 5/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.5570 - loss: 1.0764 - val_accuracy: 0.5511 - val_loss: 1.0802
Epoch 6/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.5734 - loss: 1.0399 - val_accuracy: 0.5566 - val_loss: 1.0755
Epoch 7/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.5950 - loss: 0.9999 - val_accuracy: 0.5513 - val_loss: 1.1002
Epoch 8/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.6038 - loss: 0.9789 - val_a

# Model Prediction

In [14]:
y_pred = model.predict(X_test.to_numpy())
y_pred = np.argmax(y_pred, axis=1)
y_test = np.argmax(y_test, axis=1)

119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [15]:
print(classification_report(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

           0       0.66      0.65      0.65       592
           1       0.74      0.64      0.69       599
           2       0.57      0.53      0.55      1041
           3       0.65      0.75      0.70       619
           4       0.55      0.59      0.57       947

    accuracy                           0.62      3798
   macro avg       0.64      0.63      0.63      3798
weighted avg       0.62      0.62      0.62      3798

0.6162300272194561
